<a href="https://colab.research.google.com/github/walevez/ProcesamientoDelHabla/blob/main/CHAVEZ_WALTER_PH_TP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_lg --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





## Desarrollo de las consignas:

### 1) Creacion del dataset.
Este dataset es para la a atencion a clientes de un molino harinero.

In [ ]:
dataset = {
    'hola': 'Hola, buen día. ¿En qué puedo ayudarte?',
    '¿Quisiera saber los requisitos para comprar?': 'Para poder comprar, debe estar inscripto como Responsable Inscripto.',
    '¿Qué productos venden?': 'Harinas 000 x 1 kg, Harina 0000 x 1 kg y Pellet de trigo x tonelada.',
    '¿Qué precio tienen las harinas trigo?': 'El precio de las harinas 000 es de $1.000 + IVA, harinas 0000 es de $1.500 + IVA.',
    '¿Cuál es la alicuota del IVA que se aplica?': 'La alicuota del IVA es para las harinas del 10,5% y para el subproducto 21%',
    '¿Qué precio tiene el Pellet de trigo por tonelada?': 'El precio del Pellet de trigo por tonelada es $5.000 + IVA.',
    '¿Qué demoras tienen para la entrega?': 'La demora es de 2 a 5 días, después de realizar el pedido.',
    '¿Cuál es la cantidad mínima de harina que se puede comprar?': 'La cantidad mínima que se puede comprar es de 10.000 unidades.',
    '¿Cuál es la cantidad mínima de Pellet que se puede comprar?': 'La cantidad mínima que se puede comprar es de 1 tonelada.',
    '¿Se realizan envíos?': 'Por el momento no se hacen envíos.',
    '¿Cuál es la dirección de del local?': 'La dirección es: Calle Falsa 1234.',
    '¿Cuáles son los medios de pago?': 'Los medios de pago son los siguientes: Efectivo, Transferencia y Cheques.',
    '¿Son agentes de percepción?': 'No, no somos agentes de percepción.',
    '¿Cuáles son los días de atención?': 'Los días de atención son de lunes a viernes.',
    '¿Cuáles son los horarios de atención?': 'Los horarios de atención son de 8 a 16, horario corrido.',
    '¿Cuáles son los canales de comunicación alternativos?': 'Puede llamarnos al +555 1234567798 o escribirnos por email a contacto@elmailnoexiste.com.ar',
    'Necesito hablar con alguien': 'Espere por favor, pronto será atendido.',
    '¿Reciben dólares como medio de pago?': 'Sí, recibimos dólares como medio de pago, valor billete vendedor Banco Central.',
    '¿Cuáles son los requisitos para calificar una cuenta corriente?': 'Por el momento no estamos habilitando cuentas corrientes.',
    '¿A qué email puedo enviar un currículum?': 'Para enviar tu currículum debe hacerse a este email: rrhh@elmailnoexiste.com.ar',
    '¿La empresa está atendiendo los feriados?': 'No, la empresa no atiende en feriados, solo los días hábiles.'
}

**Procesamiento de texto, de las llaves y valores del diccionario.**

In [ ]:
import re
import spacy

# cargo el modelo mas grande espanol de Spacy
nlp = spacy.load('es_core_news_lg')

# creo una funcion para quitar los signo extranos y los stopwords del documento
def limpiar_texto(texto):
    doc = nlp(texto)
    palabras = [token.text.lower() for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(palabras)

# Almaceno un dos listas las preguntas y respuestas
preguntas_limpias = [limpiar_texto(p) for p in dataset.keys()]
respuestas_limpias = [limpiar_texto(r) for r in dataset.values()]

###  2) Creacion del chatbot utilizando TFIDF y similitud del coseno.

In [ ]:
# vectorizo las preguntas ya limpias
vectorizador = TfidfVectorizer()
preguntas_tfidf = vectorizador.fit_transform(preguntas_limpias)

def chatbot_tfidf(pregunta_usuario, umbral=0.57):
    # Uso la funcion limpiar_texto para limpiar la pregunta del usuario.
    pregunta_limpia = limpiar_texto(pregunta_usuario)

    # Vectorizo el input del usuario
    pregunta_vector = vectorizador.transform([pregunta_limpia])

    # Similitud del coseno
    similitudes = cosine_similarity(pregunta_vector, preguntas_tfidf)
    indice = similitudes.argmax()
    score = similitudes[0][indice]

    # Aca condiciono si el score de la respuesta es menor al ajuste del umbral que determine
    # es decir no hay similitudes del coseno que se acerquen a esa pregunta.
    if score < umbral:
        return "No entendí la pregunta. ¿Podés reformularla?"

    # Devuelvo la respuesta original del dataset
    return list(dataset.values())[indice]

### 3) Creacion del otro chatbot utilizando embeddings de Spacy.

Elegi Spacy como modelo preentredo, probe los modelo sm md y lg, me quede con el lg porque fue el que me dio mayor coincidencia a la hora de responder las preguntas. Es un modelo grande pero mas preciso.

In [ ]:
import numpy as np

def obtener_embedding(texto):
    doc = nlp(texto)
    # Promediamos los vectores de las palabras que tienen vector
    vectores = [token.vector for token in doc if token.has_vector]
    if not vectores:
        return np.zeros(nlp.vocab.vectors_length)
    return np.mean(vectores, axis=0)

# Precalculamos los embeddings de las preguntas limpias
embeddings_preguntas = np.array([obtener_embedding(p) for p in preguntas_limpias])

def chatbot_embeddings(pregunta_usuario, umbral=0.62):
    pregunta_limpia = limpiar_texto(pregunta_usuario)
    emb_usuario = obtener_embedding(pregunta_limpia)

    # Similitud del coseno
    similitudes = cosine_similarity([emb_usuario], embeddings_preguntas)
    indice = similitudes.argmax()
    score = similitudes[0][indice]

    if score < umbral:
        return "No entendí la pregunta. ¿Podés reformularla?"

    return list(dataset.values())[indice]

### 4) Chatbots funcionando

In [ ]:
preguntas_prueba = [
    "hola",
    "cuanto cuestan la harinas",
    "realizan envios?",
    "cual es el minima de pellet",
    "puedo pagar con dolares",
    "a que mail mando el curriculum",
    "atienden los feriados?",
    "quiero hablar con alguien",
    "que productos tienen",
    "requisitos para comprar",
    'el grillo tiene hambre',
    'me podes pasar el precio de spacexAI?',
    'la harina es engrudo?'
]

print("PREGUNTAS CON +++++++++++ TF-IDF ++++++++++++++")
for p in preguntas_prueba:
    print(f"P: {p}")
    print(f"R: {chatbot_tfidf(p)}\n")

print("PREGUNTAS CON +++++++++++ SPACY ++++++++++++++")
for p in preguntas_prueba:
    print(f"P: {p}")
    print(f"R: {chatbot_embeddings(p)}\n")

### 5) Conclusiones:

Me gusto mucho realizar este trabajo, me encontre conceptos y tecnicas que utilice en el TP1, lo utilice a este como ayuda de memoria para realizar la limpieza del texto para luego poder vectorizar.

El primer chatbot lo realice con TF-IDF, con este vectorice las preguntas del usuario o cliente y luego mediante la similitud del coseno de este vector con todas las respustas (ya vectorizadas) del dataset el codigo busca cual se acerca mas a 1 y la escoje para mostrar, en la funcion hay un segundo parametro que es el umbral, aca es donde ajusto que tan preciso va ser el modelo y que toma como pregunta valida.

En el segundo chatbot lo realice con Spacy utilizando el metodo embeddings, asi como en TF-IDF vectorice los documentos, aca promedie los vectores de las preguntas del usuario y de las preguntas del dataset y procedi a realizar el mismo procedimiento con la similitud del coseno.

** Diferencias entre modelos **
Note que TF-IDF no es tan eficiente en relacion a Spacy,
los resultados fueron:

** SPACY **
10/10 respuestas con sentido bien contestadas.

3/3 respuestas trampas (sin sentido) no contestadas.

con un ajuste 0.62 en el umbral, abajo de eso el modelo empieza a reponder las preguntas sin sentidos con respuestas del dataset, arriba de eso el modelo empieza a dejar de responder las preguntas bien formuladas.

** TF-IDF **
7/10 respuestas con sentido bien contestadas.

3/3 respuestas trampas (sin sentido) no contestadas.

con un ajuste 0.57 en el umbral, abajo de eso el modelo empieza a reponder las preguntas sin sentidos con respuestas del dataset, arriba de eso el modelo deja de responder mas preguntas con sentido.


**Conclusion general:**

El modelo basado en embeddings de spaCy tuvo mejor desempenio en este dataset, especialmente con consultas informales o con errores de escritura. TF-IDF es mas sensible a la coincidencia exacta de palabras y requiere que la pregunta del usuario sea mas similar a las del dataset.

In [ ]:
# @title
# prompt: necesito un cuadro comparativo que muestre el resultado de estas preguntas, usa campos para que quede mejor presentado.
import pandas as pd
from IPython.display import display

data = {
    "Pregunta": [
        "hola",
        "cuanto cuestan la harinas",
        "realizan envios?",
        "cual es el minima de pellet",
        "puedo pagar con dolares",
        "a que mail mando el curriculum",
        "atienden los feriados?",
        "quiero hablar con alguien",
        "que productos tienen",
        "requisitos para comprar",
        "el grillo tiene hambre",
        "me podes pasar el precio de spacexAI?",
        "la harina es engrudo?"
    ],
    "Respuesta TF-IDF": [
        "Hola, buen día. ¿En qué puedo ayudarte?",
        "El precio de las harinas 000 es de 1.000 + IVA, harinas 0000 es de 1.500 + IVA.",
        "Por el momento no se hacen envíos.",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No, la empresa no atiende en feriados, solo los días hábiles.",
        "Espere por favor, pronto será atendido.",
        "Harinas 000 x 1 kg, Harina 0000 x 1 kg y Pellet de trigo x tonelada.",
        "Para poder comprar, debe estar inscripto como Responsable Inscripto.",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?"
    ],
    "Status TF-IDF": [
        "OK", "OK", "OK", "BAD", "BAD", "BAD", "OK", "OK", "OK", "OK", "OK", "OK", "OK"
    ],
    "Respuesta Spacy": [
        "Hola, buen día. ¿En qué puedo ayudarte?",
        "El precio de las harinas 000 es de 1.000 + IVA, harinas 0000 es de 1.500 + IVA.",
        "Por el momento no se hacen envíos.",
        "La cantidad mínima que se puede comprar es de 1 tonelada.",
        "Sí, recibimos dólares como medio de pago, valor billete vendedor Banco Central.",
        "Para enviar tu currículum debe hacerse a este email: rrhh@elmailnoexiste.com.ar",
        "No, la empresa no atiende en feriados, solo los días hábiles.",
        "Espere por favor, pronto será atendido.",
        "Harinas 000 x 1 kg, Harina 0000 x 1 kg y Pellet de trigo x tonelada.",
        "Para poder comprar, debe estar inscripto como Responsable Inscripto.",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?",
        "No entendí la pregunta. ¿Podés reformularla?"
    ],
    "Status Spacy": [
        "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK"
    ]
}

df = pd.DataFrame(data)

# Mostrar la tabla completa
display(df)

# Resumen rápido
print("\n--- Resumen ---")
print(f"TF-IDF: {df['Status TF-IDF'].value_counts().to_dict()}")
print(f"Spacy:  {df['Status Spacy'].value_counts().to_dict()}")